# Agentic Workspace MCP: Example Usage

This notebook demonstrates how to utilize the **Sandboxed Agentic Workspace MCP** server with the **OpenAI Agents SDK** and **OpenRouter**.

### Tech Stack:
- **MCP Server**: Sandboxed environment (Docker).
- **Orchestration**: OpenAI Agents SDK (Python).
- **Model Backend**: OpenRouter (giving you access to Gemini, Claude, etc.).

## 1. Setup Environment
We need to load our API keys from `.env` and configure the event loop for the notebook if necessary.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# load_dotenv() 
# If using a specific path:
load_dotenv(Path("..") / ".env")

print(f"OpenRouter API Key found: {bool(os.environ.get('OPENROUTER_API_KEY'))}")

OpenRouter API Key found: True


## 2. Initialize MCP Server Connection
We connect to the existing Docker-based MCP server via standard I/O (stdio).

**Security Hardening**: The container is launched with dropped capabilities, restricted memory/CPU, and no privilege escalation.

In [2]:
from agents.mcp import MCPServerStdio

# Configure the server link
# We assume the workspace root is the parent directory's 'tmp'
root_dir = Path(os.getcwd()).parent
abs_tmp_dir = str((root_dir / "tmp").resolve())

mcp_server = MCPServerStdio(
    name="Sandboxed Workspace",
    params={ 
        "command": "docker", 
        "args": [ 
            "run", "-i", "--rm", 
            "--user", "1000:1000", 
            "--security-opt", "no-new-privileges",
            "--cap-drop", "ALL",
            "--init",
            "--memory", "512m",
            "--cpus", "0.5",
            "-v", f"{abs_tmp_dir}:/workspace", 
            "agent-workspace-mcp" 
        ] 
    }, 
    client_session_timeout_seconds=60.0 
)

## 3. Define the Agent
We'll use a specialized model from OpenRouter (e.g., Gemini 2.0 Flash).

In [4]:
from agents import Agent, trace
from agents.extensions.models.litellm_model import LitellmModel

model_name = os.environ.get("DEFAULT_MODEL")

agent = Agent(
    name="WorkspaceAnalyst",
    instructions=(
        "You are a workspace operations analyst. Your goal is to explore the filesystem, "
        "audit artifacts, and perform maintenance tasks using your bash and file tools."
    ),
    model=LitellmModel(model=model_name),
    mcp_servers=[mcp_server]
)

## 4. Run the Agent (Streamed)
The following function runs the agent and prints tool calls, results, and text responses in real-time.

In [5]:
from agents import Runner, RunConfig

async def run_analysis(mission: str):
    print(f"🚀 Starting Mission: {mission}\n")
    
    async with mcp_server:
        # Enable OpenAI tracing for observability
        with trace(f"Mission: {mission[:50]}..."):
            # The Runner.run_streamed returns a result object that exposes a stream_events() generator.
            # We use await here as Jupyter inherently supports top-level async.
            stream = Runner.run_streamed(agent, mission, max_turns=15, run_config=RunConfig())
            async for event in stream.stream_events():
                if event.type == "run_item_stream_event":
                    item = event.item
                    if event.name == "tool_called":
                        print(f"\n🛠️  [TOOL CALL] {item.raw_item.name}({item.raw_item.arguments})")
                    elif event.name == "tool_output":
                        # Truncate large results for readability
                        output = str(item.output)
                        if len(output) > 200: 
                            output = output[:200] + "..."
                        print(f"✅ [RESULT] {output}")
                    elif event.name == "handoff_requested":
                        print(f"➡️  [HANDOFF] to {item.target_agent.name}")
                elif event.type == "raw_response_event":
                    from openai.types.responses import ResponseTextDeltaEvent
                    if isinstance(event.data, ResponseTextDeltaEvent):
                        print(event.data.delta, end="", flush=True)

In [ ]:

mission = "List the contents of the /workspace directory. Pick one .py file, read its metadata (size/mod time), and report your findings. Execute the file and print its output! If nothing is there, just a create a nice little python script, execute it and tell about it."
# Use top-level await (supported natively in Jupyter kernels)
await run_analysis(mission)

## 5. Advanced Editing (Dry Run & Fuzzy Matching)

The `search_and_replace` tool supports **Dry Run** mode and **Fuzzy Whitespace Matching**. 
- **Dry Run**: Preview changes as a unified diff before applying them.
- **Fuzzy Matching**: Matches code even if the agent gets the leading/trailing whitespace slightly wrong.
- **Indentation Preservation**: Automatically rebases the replacement text to match the file's indentation level.

In [ ]:
mission = """
1. Create 'logic.py' with a few python functions that have messy indentation.
2. Use search_and_replace with dry_run=True to fix the functions. 
3. Examine the diff, then apply the change with dry_run=False.
"""

await run_analysis(mission)

## 6. Coding anf Eile writing


In [6]:
mission = """
1. Write a python script to retrieve the site https://edwarddonner.com/ .
2. Save the website and convert it to markdown in a second script and save this markdown file too.
3. Install packages if necessary, use available bash tools.
"""

await run_analysis(mission)

🚀 Starting Mission: 
1. Write a python script to retrieve the site https://edwarddonner.com/ .
2. Save the website and convert it to markdown in a second script and save this markdown file too.
3. Install packages if necessary, use available bash tools.



🛠️  [TOOL CALL] run_bash({"command":"uv init website_fetcher && cd website_fetcher && uv add requests beautifulsoup4 html2text"})
✅ [RESULT] {'type': 'text', 'text': '[Exit code: 0]\nInitialized project `website-fetcher` at `/workspace/website_fetcher`\nUsing CPython 3.14.4 interpreter at: /usr/local/bin/python3.14\nCreating virtual enviro...

🛠️  [TOOL CALL] write_file({"content":"import requests\nimport os\n\ndef fetch_website(url, output_file):\n    print(f\"Fetching {url}...\")\n    try:\n        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})\n        response.raise_for_status()\n        \n        with open(output_file, 'w', encoding='utf-8') as f:\n            f.write(response.text)\n        print(f\"Success